In [0]:
# Librerias a usar 
from pyspark.sql import functions as F


## Funcion de limpieza de de Na

In [0]:
def remove_na_columns_threshold(df, porc=0.5):
    # Cantidad total de filas
    total_filas = df.count()

    # Calcular porcentaje de valores nulos por columna
    na_percent = {
        col: df.filter(F.col(col).isNull()).count() / total_filas
        for col in df.columns
    }

    # Conservar columnas cuyo porcentaje de NA sea <= al umbral
    columnas_validas = [
        col for col, porcentaje in na_percent.items()
        if porcentaje <= porc
    ]

    return df.select(columnas_validas)

In [0]:
# Leer la tabla desde Databricks
Ocupados = spark.table("workspace.default.ocupados")

Ocupados = Ocupados.withColumnRenamed(
    "regis", "regis_ocupados"
)
# Aplicar limpieza
Ocupados = remove_na_columns_threshold(Ocupados, 0.35)


## Segundo filtro de solo jefes de hogar

In [0]:
# 1. Filtrar por jefe de hogar (P6050 == "1")
Caracteristicas=spark.table("workspace.default.caracteristicas")
Caracteristicas = Caracteristicas.filter(
    F.col("P6050") == "1"
)

# 2. Eliminar columnas que tengan valores nulos
Caracteristicas = remove_na_columns_threshold(
    Caracteristicas,
    porc=0
)

# 3. Visualizar resultado
display(Caracteristicas)

## Join de las tablas

In [0]:
# Columnas para unir las dos tablas
columnas_join = ["DIRECTORIO","PERIODO","MES","PER","DPTO","FEX_C18","HOGAR","CLASE","ORDEN","SECUENCIA_P"]

# Unir las tablas
Datos = Ocupados.join(Caracteristicas,
    on=columnas_join,
    how="inner"
)


In [0]:
Datos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("datos")

In [0]:
# Columnas que se van a eliminar
datos = spark.table("workspace.default.datos")

columnas_eliminar = [
    "P3044S2", "P6850", "P6920", "P6990", "P9450",
    "P7028", "P1880", "P7090", "P7130", "P7170S1",
    "P7170S5", "P7170S6", "P7180", "P1881", "P1882",
    "P7240", "OCI", "RAMA2D_R4", "RAMA4D_R4",
    "OFICIO_C8", "PT", "P6016", "P6050", "P2057",
    "P2061", "P6090", "P1906S1", "P1906S2",
    "P1906S3", "P1906S4", "P1906S5", "P1906S6",
    "P1906S7", "P1906S8", "P6160", "P3042S1"
]

# Eliminar las columnas
datos = datos.drop(*columnas_eliminar)
